# 8.3 Enhancing Predictive Models with Survey Data I — Code Brief

## Key Concepts

- Loads the tuned-hyperparameter XGBoost model from Module 3, then re-fits it on the richer survey-enhanced feature set (`ML_SURVEY_MASTER_TRAIN/TEST.csv` — academic + demographic + TEXT_PC1...33).
- Standard classification metrics: accuracy, precision, recall, F1, confusion matrix, ROC-AUC.
- Feature importance grouped into Academic performance / Demographics / Survey-Text components to answer: did survey/text data add signal beyond academics?

In [ ]:
# If needed, install xgboost (uncomment if your environment does not already include it)
# !pip -q install xgboost


In [ ]:
import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, RocCurveDisplay
)

import matplotlib.pyplot as plt
import pickle


## Prepare: Load Training and Testing Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
filepath = '/content/drive/MyDrive/IR ML Cert/MLCert Course 3/Course 3 Data/'

train_df = pd.read_csv(filepath + 'ML_SURVEY_MASTER_TRAIN.csv')
test_df = pd.read_csv(filepath + 'ML_SURVEY_MASTER_TEST.csv')

print("Training shape:", train_df.shape)
print("Testing shape: ", test_df.shape)

train_df.head()

In [ ]:
# Create binary target
train_df['DEPARTED'] = (train_df['SEM_3_STATUS'] != 'E').astype(int)
test_df['DEPARTED'] = (test_df['SEM_3_STATUS'] != 'E').astype(int)

In [ ]:
# Separate features and target
X_train = train_df.drop(columns=['DEPARTED','SEM_3_STATUS'])
y_train = train_df['DEPARTED']

X_test  = test_df.drop(columns=['DEPARTED','SEM_3_STATUS'])
y_test  = test_df['DEPARTED']

print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_test: ", X_test.shape,  "| y_test: ", y_test.shape)


## Train: XGBoost Model

In [ ]:
# Load the model from the pickle file
tree_models_path = '/content/drive/MyDrive/Applied-Data-Analytics-For-Higher-Education-Course-3/models/'
filename1 = f'{tree_models_path}xgb_tuned_f1.pkl'
xgb_model = pickle.load(open(filename1, 'rb'))

In [ ]:
xgb_model.fit(X_train, y_train)


## Evaluate: Model Performance on Testing Data

In [ ]:
# Predictions
y_pred = xgb_model.predict(X_test)

# Predicted probabilities (useful for ROC AUC and risk scoring)
y_proba = xgb_model.predict_proba(X_test)[:, 1] if hasattr(xgb_model, "predict_proba") else None

# Metrics
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print(f"Accuracy:  {acc:.3f}")
print(f"Precision: {prec:.3f}")
print(f"Recall:    {rec:.3f}")
print(f"F1-score:  {f1:.3f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))


In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title("Confusion Matrix: XGBoost on SEM_3_STATUS")
plt.show()


In [ ]:
# ROC AUC (only if binary target and probabilities available)
if y_proba is not None and len(np.unique(y_test)) == 2:
    auc = roc_auc_score(y_test, y_proba)
    print(f"ROC AUC: {auc:.3f}")
    RocCurveDisplay.from_predictions(y_test, y_proba)
    plt.title("ROC Curve: XGBoost on SEM_3_STATUS")
    plt.show()
else:
    print("ROC AUC skipped (requires binary target and predict_proba).")


## Interpret: What Features Mattered Most?

In [ ]:
# Feature importance from the trained model
importances = xgb_model.feature_importances_
feature_names = X_train.columns

fi = (pd.DataFrame({"feature": feature_names, "importance": importances})
        .sort_values("importance", ascending=False)
        .reset_index(drop=True))

fi.head(20)


In [ ]:
# Plot top 20 features
top_n = 20
fi_top = fi.head(top_n).iloc[::-1]

plt.figure(figsize=(8, 6))
plt.barh(fi_top["feature"], fi_top["importance"])
plt.title(f"Top {top_n} Feature Importances (XGBoost)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()


## Interpret by Feature Group: Academic vs Demographics vs Survey/Text

In [ ]:
# Define feature groups using name-based rules
academic_prefixes = ["HS_GPA", "GPA_", "DFW_RATE_", "UNITS_ATTEMPTED_"]
demo_prefixes = ["GENDER_", "RACE_ETHNICITY_", "FIRST_GEN_STATUS_"]
text_prefixes = ["TEXT_PC"]  # update if you have other survey-derived prefixes (e.g., SURVEY_, Q_, SCALE_)

def assign_group(col):
    if any(col.startswith(p) for p in academic_prefixes):
        return "Academic performance"
    if any(col.startswith(p) for p in demo_prefixes):
        return "Demographics"
    if any(col.startswith(p) for p in text_prefixes):
        return "Survey/Text components"
    # Anything else: keep visible rather than hiding it
    return "Other (check)"

fi["group"] = fi["feature"].apply(assign_group)

group_importance = (fi.groupby("group")["importance"]
                      .sum()
                      .sort_values(ascending=False)
                      .reset_index())

group_importance


In [ ]:
import plotly.express as px

# Plot group importance
fig = px.bar(group_importance,
             x="group",
             y="importance",
             title="Total Feature Importance by Group (XGBoost)",
             labels={"group": "Feature Group", "importance": "Total Importance (sum)"})
fig.update_layout(xaxis_tickangle=-20)
fig.show()

## Going Deeper: Which Survey/Text Components Matter Most?

In [ ]:
# Top survey/text components (customize this filter if you have other survey prefixes)
survey_text_top = (fi[fi["group"] == "Survey/Text components"]
                   .sort_values("importance", ascending=False)
                   .head(15)
                   .reset_index(drop=True))

survey_text_top
